In [1]:
from Ingestor import Ingestor
from embedder import Embedder
from retriever import Retriever
from vector_db import VectorStore

c:\Users\Lenovo\Desktop\Rag_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evidence_for(answers):
    """Text evidence paragraphs for one question (tables and figures excluded)."""
    evidence = set()
    for answer in answers["answer"]:
        for e in answer["evidence"]:
            if not e.startswith("FLOAT SELECTED"):
                evidence.add(e)
    return evidence


def run_retrieval(dataset, num_papers, ret, methods=("dense", "keyword", "hybrid"), k=5):
    """Step 1: for each method, record the ranks of the correct chunks for every question."""
    records = {method: [] for method in methods}

    for paper in dataset.select(range(num_papers)):
        qas = paper["qas"]
        for question, answers in zip(qas["question"], qas["answers"]):
            evidence = evidence_for(answers)
            if not evidence:
                continue  # unanswerable or table-only: nothing to find

            for method in methods:
                chunks = ret.search(question, k=k, paper_id=paper["id"], method=method)
                correct_ranks = [
                    rank for rank, chunk in enumerate(chunks, start=1)
                    if chunk["text"] in evidence
                ]
                records[method].append({
                    "correct_ranks": correct_ranks,
                    "n_evidence": len(evidence),
                })

    return records


In [3]:
def compute_metrics(question_records, ks=(1, 5, 10, 20)):
    """Metrics at several cutoffs, from records retrieved with k = max(ks)."""
    n = len(question_records)
    results = {"questions": n}

    for k in ks:
        hit = mrr = recall = precision = 0.0
        for r in question_records:
            ranks = [rank for rank in r["correct_ranks"] if rank <= k]  # only what's in the top k
            if ranks:
                hit += 1
                mrr += 1 / ranks[0]
            recall += len(ranks) / r["n_evidence"]
            precision += len(ranks) / k

        results[f"hit@{k}"] = hit / n
        results[f"MRR@{k}"] = mrr / n
        results[f"recall@{k}"] = recall / n
        results[f"precision@{k}"] = precision / n

    return results

In [4]:
ingestor = Ingestor()
embedder = Embedder()
vectorstore = VectorStore(dimension=embedder.dimension)
ret = Retriever(embedder, vectorstore)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 993.19it/s]
c:\Users\Lenovo\Desktop\Rag_Project\embedder.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


In [5]:
ingestor.ingest(embedder, vectorstore) 

print(vectorstore.count())

281 papers -> 13266 chunks


Batches: 100%|██████████| 415/415 [26:27<00:00,  3.82s/it] 


Embedded in 1589.0s
Stored: {'chunks': 13266, 'vec_chunks': 13266, 'fts_chunks': 13266}
{'chunks': 13266, 'vec_chunks': 13266, 'fts_chunks': 13266}


In [6]:
ks = (1, 5, 10, 20)

dataset = ingestor.load_qasper_validation()
records = run_retrieval(dataset, 281, ret, k=max(ks))   # retrieve the top 20 once

for method, question_records in records.items():
    m = compute_metrics(question_records, ks)
    print(f"\n{method}  ({m['questions']} questions)")
    for k in ks:
        print(f"  @{k:<3} hit={m[f'hit@{k}']:.3f}  MRR={m[f'MRR@{k}']:.3f}  "
              f"recall={m[f'recall@{k}']:.3f}  precision={m[f'precision@{k}']:.3f}")


dense  (892 questions)
  @1   hit=0.287  MRR=0.287  recall=0.195  precision=0.287
  @5   hit=0.673  MRR=0.428  recall=0.515  precision=0.167
  @10  hit=0.825  MRR=0.448  recall=0.679  precision=0.117
  @20  hit=0.938  MRR=0.457  recall=0.833  precision=0.076

keyword  (892 questions)
  @1   hit=0.224  MRR=0.224  recall=0.155  precision=0.224
  @5   hit=0.628  MRR=0.371  recall=0.477  precision=0.154
  @10  hit=0.796  MRR=0.394  recall=0.652  precision=0.111
  @20  hit=0.920  MRR=0.402  recall=0.815  precision=0.073

hybrid  (892 questions)
  @1   hit=0.290  MRR=0.290  recall=0.196  precision=0.290
  @5   hit=0.697  MRR=0.442  recall=0.537  precision=0.174
  @10  hit=0.836  MRR=0.460  recall=0.693  precision=0.118
  @20  hit=0.957  MRR=0.469  recall=0.855  precision=0.077


In [7]:
# testing the generator on 1 question

from generator import Generator

generator = Generator()


In [8]:

question = "What language pairs are explored?"
paper_id = "1912.01214"

chunks = ret.search(question, k=5, paper_id=paper_id)
answer = generator.generate(question, chunks)

print(answer)
print("Sections used:", [c["section"] for c in chunks])

The following language pairs are explored:  
- French-English-Spanish (Fr-En-Es)  
- German-English-French (De-En-Fr)  
- Romanian-English-German (Ro-En-De)  
- Arabic-English-Spanish (Ar-En-Es)  
- Arabic-English-Russian (Ar-En-Ru)  
- Spanish-English-Russian (Es-En-Ru)  

These pairs are evaluated using English as the pivot language, with zero-shot translation settings [Section: Experiments ::: Setup ::: Datasets].
Sections used: ['Experiments ::: Setup ::: Datasets.', 'Approach ::: Bridge Language Model Pretraining', 'Introduction', 'Related Work', 'Experiments ::: Setup ::: Datasets.']


In [9]:
print(generator.model)

qwen3:4b-instruct-2507-q4_K_M


In [10]:
import time, httpx

start = time.perf_counter()
r = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b-instruct-2507-q4_K_M",
        "messages": [{"role": "user", "content": "Say hello in one sentence."}],
        "stream": False,
        "options": {"num_ctx": 4096, "num_predict": 50},
    },
    timeout=None,
)
print(f"{time.perf_counter() - start:.1f}s", r.json()["message"]["content"])

6.4s Hello! 😊
